# Cell 1. 라이브러리 및 데이터 로드

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from scipy.stats import (
    shapiro,
    levene,
    ttest_ind,
    mannwhitneyu,
    chi2_contingency
)

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 200)

# 경로는 네 환경에 맞게 수정
DATA_PATH = Path("results/result_sample_shorts_all_for_video_agent_fixed.csv")

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print("데이터 크기:", df.shape)

데이터 크기: (200, 75)


# Cell 2. 분석 변수 정의

In [3]:
numeric_cols = [
    "avg_brightness",
    "avg_blue",
    "avg_green",
    "avg_red",
    "person_ratio",
    "face_ratio",
    "text_ratio",
]

categorical_cols = [
    "production_quality",
    "lighting_style",
    "color_mood",
    "editing_pace",
    "motion_graphic",
    "video_format",
    "first_3sec",
    "background_style",
]

# 실제 존재하는 컬럼만 사용
numeric_cols = [col for col in numeric_cols if col in df.columns]
categorical_cols = [col for col in categorical_cols if col in df.columns]

print("수치형 변수:", numeric_cols)
print("범주형 변수:", categorical_cols)

# 성공/실패 라벨 확인
display(df["domain"].value_counts())
display(df["success_label"].value_counts())
display(pd.crosstab(df["domain"], df["success_label"]))

수치형 변수: ['avg_brightness', 'avg_blue', 'avg_green', 'avg_red', 'person_ratio', 'face_ratio', 'text_ratio']
범주형 변수: ['production_quality', 'lighting_style', 'color_mood', 'editing_pace', 'motion_graphic', 'video_format', 'first_3sec', 'background_style']


domain
FnB    100
IT     100
Name: count, dtype: int64

success_label
fail       110
success     90
Name: count, dtype: int64

success_label,fail,success
domain,,
FnB,52,48
IT,58,42


# Cell 3. 정규성 검정: Shapiro-Wilk test

In [4]:
normality_rows = []

for domain in df["domain"].unique():
    for label in ["success", "fail"]:
        temp = df[
            (df["domain"] == domain) &
            (df["success_label"] == label)
        ].copy()
        
        for col in numeric_cols:
            values = temp[col].dropna()
            
            if len(values) < 3:
                normality_rows.append({
                    "domain": domain,
                    "success_label": label,
                    "feature": col,
                    "n": len(values),
                    "shapiro_stat": np.nan,
                    "shapiro_p": np.nan,
                    "normality_result": "표본 부족"
                })
                continue
            
            stat, p = shapiro(values)
            
            normality_rows.append({
                "domain": domain,
                "success_label": label,
                "feature": col,
                "n": len(values),
                "shapiro_stat": round(stat, 4),
                "shapiro_p": round(p, 4),
                "normality_result": "정규성 만족 가능" if p >= 0.05 else "정규성 불만족 가능"
            })

normality_df = pd.DataFrame(normality_rows)

display(
    normality_df.sort_values(
        ["domain", "feature", "success_label"]
    )
)

,domain,success_label,feature,n,shapiro_stat,shapiro_p,normality_result
8,FnB,fail,avg_blue,52,0.9779,0.4423,정규성 만족 가능
1,FnB,success,avg_blue,48,0.9611,0.1123,정규성 만족 가능
7,FnB,fail,avg_brightness,52,0.9645,0.1221,정규성 만족 가능
0,FnB,success,avg_brightness,48,0.9677,0.2052,정규성 만족 가능
9,FnB,fail,avg_green,52,0.9713,0.2405,정규성 만족 가능
2,FnB,success,avg_green,48,0.9721,0.3041,정규성 만족 가능
10,FnB,fail,avg_red,52,0.9513,0.0332,정규성 불만족 가능
3,FnB,success,avg_red,48,0.9628,0.1313,정규성 만족 가능
12,FnB,fail,face_ratio,52,0.8172,0.0000,정규성 불만족 가능
5,FnB,success,face_ratio,48,0.8370,0.0000,정규성 불만족 가능


# Cell 4. 정규성 검정 요약

In [5]:
normality_summary = (
    normality_df
    .groupby(["domain", "feature", "normality_result"])
    .size()
    .reset_index(name="count")
)

display(normality_summary)

print("정규성 불만족 가능 변수")
display(
    normality_df[
        normality_df["normality_result"] == "정규성 불만족 가능"
    ].sort_values(["domain", "feature", "success_label"])
)

,domain,feature,normality_result,count
0,FnB,avg_blue,정규성 만족 가능,2
1,FnB,avg_brightness,정규성 만족 가능,2
2,FnB,avg_green,정규성 만족 가능,2
3,FnB,avg_red,정규성 만족 가능,1
4,FnB,avg_red,정규성 불만족 가능,1
5,FnB,face_ratio,정규성 불만족 가능,2
6,FnB,person_ratio,정규성 불만족 가능,2
7,FnB,text_ratio,정규성 불만족 가능,2
8,IT,avg_blue,정규성 만족 가능,1
9,IT,avg_blue,정규성 불만족 가능,1


정규성 불만족 가능 변수


,domain,success_label,feature,n,shapiro_stat,shapiro_p,normality_result
10,FnB,fail,avg_red,52,0.9513,0.0332,정규성 불만족 가능
12,FnB,fail,face_ratio,52,0.8172,0.0000,정규성 불만족 가능
5,FnB,success,face_ratio,48,0.8370,0.0000,정규성 불만족 가능
11,FnB,fail,person_ratio,52,0.8449,0.0000,정규성 불만족 가능
4,FnB,success,person_ratio,48,0.7367,0.0000,정규성 불만족 가능
13,FnB,fail,text_ratio,52,0.9478,0.0236,정규성 불만족 가능
6,FnB,success,text_ratio,48,0.9141,0.0019,정규성 불만족 가능
22,IT,fail,avg_blue,58,0.9539,0.0275,정규성 불만족 가능
21,IT,fail,avg_brightness,58,0.9481,0.0149,정규성 불만족 가능
23,IT,fail,avg_green,58,0.9430,0.0088,정규성 불만족 가능


# Cell 5. 등분산성 검정: Levene test

In [6]:
variance_rows = []

for domain in df["domain"].unique():
    temp = df[df["domain"] == domain].copy()
    
    success_df = temp[temp["success_label"] == "success"]
    fail_df = temp[temp["success_label"] == "fail"]
    
    for col in numeric_cols:
        success_values = success_df[col].dropna()
        fail_values = fail_df[col].dropna()
        
        if len(success_values) < 2 or len(fail_values) < 2:
            variance_rows.append({
                "domain": domain,
                "feature": col,
                "success_n": len(success_values),
                "fail_n": len(fail_values),
                "levene_stat": np.nan,
                "levene_p": np.nan,
                "variance_result": "표본 부족"
            })
            continue
        
        stat, p = levene(success_values, fail_values)
        
        variance_rows.append({
            "domain": domain,
            "feature": col,
            "success_n": len(success_values),
            "fail_n": len(fail_values),
            "levene_stat": round(stat, 4),
            "levene_p": round(p, 4),
            "variance_result": "등분산 가정 가능" if p >= 0.05 else "등분산 가정 어려움"
        })

variance_df = pd.DataFrame(variance_rows)

display(
    variance_df.sort_values(["domain", "feature"])
)

,domain,feature,success_n,fail_n,levene_stat,levene_p,variance_result
1,FnB,avg_blue,48,52,0.0096,0.9222,등분산 가정 가능
0,FnB,avg_brightness,48,52,0.2169,0.6425,등분산 가정 가능
2,FnB,avg_green,48,52,0.5227,0.4714,등분산 가정 가능
3,FnB,avg_red,48,52,0.8203,0.3673,등분산 가정 가능
5,FnB,face_ratio,48,52,11.6313,0.0009,등분산 가정 어려움
4,FnB,person_ratio,48,52,4.4839,0.0367,등분산 가정 어려움
6,FnB,text_ratio,48,52,0.0222,0.8820,등분산 가정 가능
8,IT,avg_blue,42,58,2.3921,0.1252,등분산 가정 가능
7,IT,avg_brightness,42,58,2.1094,0.1496,등분산 가정 가능
9,IT,avg_green,42,58,1.7469,0.1893,등분산 가정 가능


# Cell 6. 등분산성 검정 요약

In [7]:
print("등분산 가정 어려운 변수")
display(
    variance_df[
        variance_df["variance_result"] == "등분산 가정 어려움"
    ].sort_values(["domain", "feature"])
)

등분산 가정 어려운 변수


,domain,feature,success_n,fail_n,levene_stat,levene_p,variance_result
5,FnB,face_ratio,48,52,11.6313,0.0009,등분산 가정 어려움
4,FnB,person_ratio,48,52,4.4839,0.0367,등분산 가정 어려움


# Cell 7. 정규성 + 등분산성 요약표 만들기

In [8]:
# 성공/실패별 정규성 결과를 wide 형태로 변환
normality_wide = normality_df.pivot_table(
    index=["domain", "feature"],
    columns="success_label",
    values=["shapiro_p", "normality_result"],
    aggfunc="first"
).reset_index()

# 컬럼명 정리
normality_wide.columns = [
    "_".join([str(x) for x in col if x != ""]).strip("_")
    for col in normality_wide.columns
]

# 등분산성 결과와 병합
assumption_summary = variance_df.merge(
    normality_wide,
    on=["domain", "feature"],
    how="left"
)

# 컬럼명 순서 정리
assumption_summary = assumption_summary[
    [
        "domain",
        "feature",
        "success_n",
        "fail_n",
        "shapiro_p_success",
        "normality_result_success",
        "shapiro_p_fail",
        "normality_result_fail",
        "levene_p",
        "variance_result",
    ]
]

display(
    assumption_summary.sort_values(["domain", "feature"])
)

,domain,feature,success_n,fail_n,shapiro_p_success,normality_result_success,shapiro_p_fail,normality_result_fail,levene_p,variance_result
1,FnB,avg_blue,48,52,0.1123,정규성 만족 가능,0.4423,정규성 만족 가능,0.9222,등분산 가정 가능
0,FnB,avg_brightness,48,52,0.2052,정규성 만족 가능,0.1221,정규성 만족 가능,0.6425,등분산 가정 가능
2,FnB,avg_green,48,52,0.3041,정규성 만족 가능,0.2405,정규성 만족 가능,0.4714,등분산 가정 가능
3,FnB,avg_red,48,52,0.1313,정규성 만족 가능,0.0332,정규성 불만족 가능,0.3673,등분산 가정 가능
5,FnB,face_ratio,48,52,0.0000,정규성 불만족 가능,0.0000,정규성 불만족 가능,0.0009,등분산 가정 어려움
4,FnB,person_ratio,48,52,0.0000,정규성 불만족 가능,0.0000,정규성 불만족 가능,0.0367,등분산 가정 어려움
6,FnB,text_ratio,48,52,0.0019,정규성 불만족 가능,0.0236,정규성 불만족 가능,0.8820,등분산 가정 가능
8,IT,avg_blue,42,58,0.0930,정규성 만족 가능,0.0275,정규성 불만족 가능,0.1252,등분산 가정 가능
7,IT,avg_brightness,42,58,0.0504,정규성 만족 가능,0.0149,정규성 불만족 가능,0.1496,등분산 가정 가능
9,IT,avg_green,42,58,0.0230,정규성 불만족 가능,0.0088,정규성 불만족 가능,0.1893,등분산 가정 가능


# Cell 8. 효과크기 함수: Cohen’s d

In [9]:
def cohens_d(x, y):
    """
    두 집단 간 평균 차이의 효과크기 계산
    success - fail 기준
    """
    x = x.dropna()
    y = y.dropna()
    
    nx = len(x)
    ny = len(y)
    
    if nx < 2 or ny < 2:
        return np.nan
    
    pooled_std = np.sqrt(
        ((nx - 1) * x.std() ** 2 + (ny - 1) * y.std() ** 2) / (nx + ny - 2)
    )
    
    if pooled_std == 0:
        return np.nan
    
    return (x.mean() - y.mean()) / pooled_std

# Cell 9. FDR 보정 함수

In [11]:
def benjamini_hochberg(p_values):
    """
    Benjamini-Hochberg FDR 보정
    입력: p-value 리스트 또는 Series
    출력: 보정 p-value array
    """
    p_values = np.array(p_values, dtype=float)
    n = len(p_values)
    
    order = np.argsort(p_values)
    ranked_p = p_values[order]
    
    adjusted = ranked_p * n / (np.arange(1, n + 1))
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    
    result = np.empty(n)
    result[order] = adjusted
    
    return result

# Cell 10. 수치형 변수 검정: Welch’s t-test + Mann-Whitney U test

In [12]:
numeric_test_rows = []

for domain in df["domain"].unique():
    temp = df[df["domain"] == domain].copy()
    
    success_df = temp[temp["success_label"] == "success"]
    fail_df = temp[temp["success_label"] == "fail"]
    
    for col in numeric_cols:
        success_values = success_df[col].dropna()
        fail_values = fail_df[col].dropna()
        
        if len(success_values) < 2 or len(fail_values) < 2:
            continue
        
        # Welch's t-test: 등분산을 가정하지 않음
        t_stat, t_p = ttest_ind(
            success_values,
            fail_values,
            equal_var=False
        )
        
        # Mann-Whitney U test: 비모수 검정
        u_stat, u_p = mannwhitneyu(
            success_values,
            fail_values,
            alternative="two-sided"
        )
        
        d = cohens_d(success_values, fail_values)
        
        numeric_test_rows.append({
            "domain": domain,
            "feature": col,
            "success_mean": round(success_values.mean(), 3),
            "fail_mean": round(fail_values.mean(), 3),
            "diff_success_minus_fail": round(success_values.mean() - fail_values.mean(), 3),
            "t_stat": round(t_stat, 4),
            "t_test_p": round(t_p, 4),
            "u_stat": round(u_stat, 4),
            "mannwhitney_p": round(u_p, 4),
            "cohens_d": round(d, 3),
            "success_n": len(success_values),
            "fail_n": len(fail_values),
        })

numeric_test_df = pd.DataFrame(numeric_test_rows)

# FDR 보정
numeric_test_df["t_test_p_fdr"] = np.nan
numeric_test_df["mannwhitney_p_fdr"] = np.nan

for domain in numeric_test_df["domain"].unique():
    mask = numeric_test_df["domain"] == domain
    
    numeric_test_df.loc[mask, "t_test_p_fdr"] = benjamini_hochberg(
        numeric_test_df.loc[mask, "t_test_p"]
    )
    
    numeric_test_df.loc[mask, "mannwhitney_p_fdr"] = benjamini_hochberg(
        numeric_test_df.loc[mask, "mannwhitney_p"]
    )

numeric_test_df["t_test_p_fdr"] = numeric_test_df["t_test_p_fdr"].round(4)
numeric_test_df["mannwhitney_p_fdr"] = numeric_test_df["mannwhitney_p_fdr"].round(4)

display(
    numeric_test_df.sort_values(
        ["domain", "mannwhitney_p"],
        ascending=[True, True]
    )
)

,domain,feature,success_mean,fail_mean,diff_success_minus_fail,t_stat,t_test_p,u_stat,mannwhitney_p,cohens_d,success_n,fail_n,t_test_p_fdr,mannwhitney_p_fdr
4,FnB,person_ratio,0.786,0.602,0.185,3.0011,0.0034,1734.0,0.0008,0.595,48,52,0.0119,0.0032
5,FnB,face_ratio,0.649,0.399,0.250,3.6647,0.0004,1724.5,0.0009,0.727,48,52,0.0028,0.0032
0,FnB,avg_brightness,103.751,124.562,-20.811,-2.3614,0.0202,878.0,0.0108,-0.471,48,52,0.0395,0.0252
2,FnB,avg_green,99.241,120.713,-21.471,-2.3172,0.0226,895.0,0.0150,-0.462,48,52,0.0395,0.0262
3,FnB,avg_red,115.789,137.047,-21.258,-2.2159,0.0290,917.0,0.0226,-0.441,48,52,0.0406,0.0316
1,FnB,avg_blue,93.767,111.943,-18.176,-1.9412,0.0551,960.0,0.0473,-0.388,48,52,0.0643,0.0552
6,FnB,text_ratio,0.712,0.652,0.060,1.2788,0.2040,1450.0,0.1618,0.256,48,52,0.2040,0.1618
10,IT,avg_red,112.947,97.780,15.167,1.6201,0.1087,1463.5,0.0871,0.325,42,58,0.1708,0.2418
8,IT,avg_blue,119.381,103.477,15.904,1.5634,0.1213,1438.0,0.1253,0.311,42,58,0.1708,0.2418
7,IT,avg_brightness,112.247,97.767,14.480,1.5609,0.1220,1419.0,0.1614,0.312,42,58,0.1708,0.2418


# Cell 11. 수치형 검정 결과 + 정규성/등분산성 병합

In [13]:
numeric_test_with_assumption_df = numeric_test_df.merge(
    assumption_summary[
        [
            "domain",
            "feature",
            "shapiro_p_success",
            "normality_result_success",
            "shapiro_p_fail",
            "normality_result_fail",
            "levene_p",
            "variance_result",
        ]
    ],
    on=["domain", "feature"],
    how="left"
)

display(
    numeric_test_with_assumption_df.sort_values(
        ["domain", "mannwhitney_p"],
        ascending=[True, True]
    )
)

,domain,feature,success_mean,fail_mean,diff_success_minus_fail,t_stat,t_test_p,u_stat,mannwhitney_p,cohens_d,success_n,fail_n,t_test_p_fdr,mannwhitney_p_fdr,shapiro_p_success,normality_result_success,shapiro_p_fail,normality_result_fail,levene_p,variance_result
4,FnB,person_ratio,0.786,0.602,0.185,3.0011,0.0034,1734.0,0.0008,0.595,48,52,0.0119,0.0032,0.0000,정규성 불만족 가능,0.0000,정규성 불만족 가능,0.0367,등분산 가정 어려움
5,FnB,face_ratio,0.649,0.399,0.250,3.6647,0.0004,1724.5,0.0009,0.727,48,52,0.0028,0.0032,0.0000,정규성 불만족 가능,0.0000,정규성 불만족 가능,0.0009,등분산 가정 어려움
0,FnB,avg_brightness,103.751,124.562,-20.811,-2.3614,0.0202,878.0,0.0108,-0.471,48,52,0.0395,0.0252,0.2052,정규성 만족 가능,0.1221,정규성 만족 가능,0.6425,등분산 가정 가능
2,FnB,avg_green,99.241,120.713,-21.471,-2.3172,0.0226,895.0,0.0150,-0.462,48,52,0.0395,0.0262,0.3041,정규성 만족 가능,0.2405,정규성 만족 가능,0.4714,등분산 가정 가능
3,FnB,avg_red,115.789,137.047,-21.258,-2.2159,0.0290,917.0,0.0226,-0.441,48,52,0.0406,0.0316,0.1313,정규성 만족 가능,0.0332,정규성 불만족 가능,0.3673,등분산 가정 가능
1,FnB,avg_blue,93.767,111.943,-18.176,-1.9412,0.0551,960.0,0.0473,-0.388,48,52,0.0643,0.0552,0.1123,정규성 만족 가능,0.4423,정규성 만족 가능,0.9222,등분산 가정 가능
6,FnB,text_ratio,0.712,0.652,0.060,1.2788,0.2040,1450.0,0.1618,0.256,48,52,0.2040,0.1618,0.0019,정규성 불만족 가능,0.0236,정규성 불만족 가능,0.8820,등분산 가정 가능
10,IT,avg_red,112.947,97.780,15.167,1.6201,0.1087,1463.5,0.0871,0.325,42,58,0.1708,0.2418,0.1410,정규성 만족 가능,0.0102,정규성 불만족 가능,0.3572,등분산 가정 가능
8,IT,avg_blue,119.381,103.477,15.904,1.5634,0.1213,1438.0,0.1253,0.311,42,58,0.1708,0.2418,0.0930,정규성 만족 가능,0.0275,정규성 불만족 가능,0.1252,등분산 가정 가능
7,IT,avg_brightness,112.247,97.767,14.480,1.5609,0.1220,1419.0,0.1614,0.312,42,58,0.1708,0.2418,0.0504,정규성 만족 가능,0.0149,정규성 불만족 가능,0.1496,등분산 가정 가능


# Cell 12. 핵심 수치형 변수만 보기

In [14]:
key_numeric_features = [
    "person_ratio",
    "face_ratio",
    "text_ratio",
]

display(
    numeric_test_with_assumption_df[
        numeric_test_with_assumption_df["feature"].isin(key_numeric_features)
    ].sort_values(["domain", "feature"])
)

,domain,feature,success_mean,fail_mean,diff_success_minus_fail,t_stat,t_test_p,u_stat,mannwhitney_p,cohens_d,success_n,fail_n,t_test_p_fdr,mannwhitney_p_fdr,shapiro_p_success,normality_result_success,shapiro_p_fail,normality_result_fail,levene_p,variance_result
5,FnB,face_ratio,0.649,0.399,0.250,3.6647,0.0004,1724.5,0.0009,0.727,48,52,0.0028,0.0032,0.0000,정규성 불만족 가능,0.0000,정규성 불만족 가능,0.0009,등분산 가정 어려움
4,FnB,person_ratio,0.786,0.602,0.185,3.0011,0.0034,1734.0,0.0008,0.595,48,52,0.0119,0.0032,0.0000,정규성 불만족 가능,0.0000,정규성 불만족 가능,0.0367,등분산 가정 어려움
6,FnB,text_ratio,0.712,0.652,0.060,1.2788,0.2040,1450.0,0.1618,0.256,48,52,0.2040,0.1618,0.0019,정규성 불만족 가능,0.0236,정규성 불만족 가능,0.8820,등분산 가정 가능
12,IT,face_ratio,0.524,0.641,-0.118,-1.7090,0.0914,1023.0,0.1727,-0.355,42,58,0.1708,0.2418,0.0000,정규성 불만족 가능,0.0000,정규성 불만족 가능,0.1144,등분산 가정 가능
11,IT,person_ratio,0.621,0.734,-0.113,-1.6594,0.1012,1055.0,0.2531,-0.350,42,58,0.1708,0.2953,0.0000,정규성 불만족 가능,0.0000,정규성 불만족 가능,0.0866,등분산 가정 가능
13,IT,text_ratio,0.807,0.769,0.037,1.2021,0.2322,1334.5,0.4118,0.233,42,58,0.2322,0.4118,0.0091,정규성 불만족 가능,0.0006,정규성 불만족 가능,0.1061,등분산 가정 가능


# Cell 13. Cramer’s V 함수

In [15]:
def cramers_v(confusion_matrix):
    """
    카이제곱 검정의 효과크기 계산
    범주형 변수 간 관계 강도 측정
    """
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    
    if n == 0:
        return np.nan
    
    min_dim = min(k - 1, r - 1)
    
    if min_dim == 0:
        return np.nan
    
    return np.sqrt((chi2 / n) / min_dim)

# Cell 14. 범주형 변수 검정: 카이제곱 + Cramer’s V + 기대빈도 확인

In [17]:
category_test_rows = []

for domain in df["domain"].unique():
    temp = df[df["domain"] == domain].copy()
    
    for col in categorical_cols:
        table = pd.crosstab(temp["success_label"], temp[col])
        
        if table.shape[0] < 2 or table.shape[1] < 2:
            continue
        
        chi2, p, dof, expected = chi2_contingency(table)
        cv = cramers_v(table)
        
        expected_df = pd.DataFrame(
            expected,
            index=table.index,
            columns=table.columns
        )
        
        min_expected = expected_df.min().min()
        low_expected_count = (expected_df < 5).sum().sum()
        low_expected_ratio = low_expected_count / expected_df.size
        
        category_test_rows.append({
            "domain": domain,
            "feature": col,
            "chi2_stat": round(chi2, 4),
            "chi2_p": round(p, 4),
            "dof": dof,
            "cramers_v": round(cv, 3),
            "n_categories": table.shape[1],
            "success_n": table.loc["success"].sum() if "success" in table.index else 0,
            "fail_n": table.loc["fail"].sum() if "fail" in table.index else 0,
            "min_expected": round(min_expected, 3),
            "low_expected_count": int(low_expected_count),
            "low_expected_ratio": round(low_expected_ratio, 3),
            "expected_freq_warning": "주의 필요" if low_expected_ratio > 0.2 else "대체로 양호"
        })

category_test_df = pd.DataFrame(category_test_rows)

# FDR 보정
category_test_df["chi2_p_fdr"] = np.nan

for domain in category_test_df["domain"].unique():
    mask = category_test_df["domain"] == domain
    category_test_df.loc[mask, "chi2_p_fdr"] = benjamini_hochberg(
        category_test_df.loc[mask, "chi2_p"]
    )

category_test_df["chi2_p_fdr"] = category_test_df["chi2_p_fdr"].round(4)

display(
    category_test_df.sort_values(
        ["domain", "chi2_p"],
        ascending=[True, True]
    )
)

,domain,feature,chi2_stat,chi2_p,dof,cramers_v,n_categories,success_n,fail_n,min_expected,low_expected_count,low_expected_ratio,expected_freq_warning,chi2_p_fdr
4,FnB,motion_graphic,12.2744,0.0022,2,0.350,3,48,52,0.96,2,0.333,주의 필요,0.0176
6,FnB,first_3sec,13.4314,0.0197,5,0.366,6,48,52,0.48,6,0.500,주의 필요,0.0788
5,FnB,video_format,16.7500,0.1155,11,0.409,12,48,52,0.48,20,0.833,주의 필요,0.3080
2,FnB,color_mood,4.4681,0.2152,3,0.211,4,48,52,0.48,2,0.250,주의 필요,0.4304
3,FnB,editing_pace,2.7292,0.4353,3,0.165,4,48,52,1.44,3,0.375,주의 필요,0.5855
0,FnB,production_quality,1.6459,0.4391,2,0.128,3,48,52,4.32,2,0.333,주의 필요,0.5855
7,FnB,background_style,1.8678,0.6003,3,0.137,4,48,52,3.36,2,0.250,주의 필요,0.6161
1,FnB,lighting_style,0.9686,0.6161,2,0.098,3,48,52,4.32,2,0.333,주의 필요,0.6161
12,IT,motion_graphic,6.4630,0.0110,1,0.254,2,42,58,16.38,0,0.000,대체로 양호,0.0880
14,IT,first_3sec,6.7947,0.0787,3,0.261,4,42,58,0.84,4,0.500,주의 필요,0.2712


# Cell 15. 핵심 범주형 변수만 보기

In [18]:
key_categorical_features = [
    "first_3sec",
    "motion_graphic",
    "editing_pace",
    "video_format",
]

display(
    category_test_df[
        category_test_df["feature"].isin(key_categorical_features)
    ].sort_values(["domain", "chi2_p"])
)

,domain,feature,chi2_stat,chi2_p,dof,cramers_v,n_categories,success_n,fail_n,min_expected,low_expected_count,low_expected_ratio,expected_freq_warning,chi2_p_fdr
4,FnB,motion_graphic,12.2744,0.0022,2,0.350,3,48,52,0.96,2,0.333,주의 필요,0.0176
6,FnB,first_3sec,13.4314,0.0197,5,0.366,6,48,52,0.48,6,0.500,주의 필요,0.0788
5,FnB,video_format,16.7500,0.1155,11,0.409,12,48,52,0.48,20,0.833,주의 필요,0.3080
3,FnB,editing_pace,2.7292,0.4353,3,0.165,4,48,52,1.44,3,0.375,주의 필요,0.5855
12,IT,motion_graphic,6.4630,0.0110,1,0.254,2,42,58,16.38,0,0.000,대체로 양호,0.0880
14,IT,first_3sec,6.7947,0.0787,3,0.261,4,42,58,0.84,4,0.500,주의 필요,0.2712
13,IT,video_format,18.4849,0.1017,12,0.430,13,42,58,0.42,19,0.731,주의 필요,0.2712
11,IT,editing_pace,5.6181,0.2295,4,0.237,5,42,58,0.42,4,0.400,주의 필요,0.3672


# Cell 16. 기대빈도 주의 필요한 범주형 변수 확인

In [19]:
print("기대빈도 주의 필요한 변수")
display(
    category_test_df[
        category_test_df["expected_freq_warning"] == "주의 필요"
    ].sort_values(["domain", "feature"])
)

기대빈도 주의 필요한 변수


,domain,feature,chi2_stat,chi2_p,dof,cramers_v,n_categories,success_n,fail_n,min_expected,low_expected_count,low_expected_ratio,expected_freq_warning,chi2_p_fdr
7,FnB,background_style,1.8678,0.6003,3,0.137,4,48,52,3.36,2,0.250,주의 필요,0.6161
2,FnB,color_mood,4.4681,0.2152,3,0.211,4,48,52,0.48,2,0.250,주의 필요,0.4304
3,FnB,editing_pace,2.7292,0.4353,3,0.165,4,48,52,1.44,3,0.375,주의 필요,0.5855
6,FnB,first_3sec,13.4314,0.0197,5,0.366,6,48,52,0.48,6,0.500,주의 필요,0.0788
1,FnB,lighting_style,0.9686,0.6161,2,0.098,3,48,52,4.32,2,0.333,주의 필요,0.6161
4,FnB,motion_graphic,12.2744,0.0022,2,0.350,3,48,52,0.96,2,0.333,주의 필요,0.0176
0,FnB,production_quality,1.6459,0.4391,2,0.128,3,48,52,4.32,2,0.333,주의 필요,0.5855
5,FnB,video_format,16.7500,0.1155,11,0.409,12,48,52,0.48,20,0.833,주의 필요,0.3080
15,IT,background_style,3.5907,0.3092,3,0.189,4,42,58,2.94,2,0.250,주의 필요,0.4123
10,IT,color_mood,1.7234,0.7865,4,0.131,5,42,58,0.84,3,0.300,주의 필요,0.7865


# Cell 17. 범주별 성공/실패 비율표 함수

In [20]:
def make_category_ratio_table(df, category_col, group_cols=["domain", "success_label"]):
    count_df = (
        df.groupby(group_cols + [category_col])
          .size()
          .reset_index(name="count")
    )
    
    total_df = (
        df.groupby(group_cols)
          .size()
          .reset_index(name="total")
    )
    
    result = count_df.merge(total_df, on=group_cols, how="left")
    result["ratio"] = (result["count"] / result["total"]).round(3)
    
    return result


def make_success_fail_category_diff(df, category_col):
    ratio_df = make_category_ratio_table(df, category_col)
    
    pivot = ratio_df.pivot_table(
        index=["domain", category_col],
        columns="success_label",
        values="ratio",
        fill_value=0
    ).reset_index()
    
    if "success" not in pivot.columns:
        pivot["success"] = 0
    if "fail" not in pivot.columns:
        pivot["fail"] = 0
    
    pivot["diff_success_minus_fail"] = (pivot["success"] - pivot["fail"]).round(3)
    
    return pivot.sort_values(
        ["domain", "diff_success_minus_fail"],
        ascending=[True, False]
    )

# Cell 18. 핵심 범주형 변수별 성공/실패 비율 차이

In [21]:
category_diff_dict = {}

for col in key_categorical_features:
    if col not in df.columns:
        continue
    
    diff_table = make_success_fail_category_diff(df, col)
    category_diff_dict[col] = diff_table
    
    print("=" * 80)
    print(col)
    display(diff_table)

first_3sec


success_label,domain,first_3sec,fail,success,diff_success_minus_fail
2,FnB,인물,0.212,0.521,0.309
3,FnB,장면,0.019,0.042,0.023
4,FnB,제품,0.173,0.167,-0.006
1,FnB,음식,0.038,0.021,-0.017
0,FnB,기업 로고,0.019,0.000,-0.019
5,FnB,텍스트,0.538,0.250,-0.288
9,IT,텍스트,0.621,0.833,0.212
8,IT,제품,0.034,0.000,-0.034
7,IT,장면,0.052,0.000,-0.052
6,IT,인물,0.293,0.167,-0.126


motion_graphic


success_label,domain,motion_graphic,fail,success,diff_success_minus_fail
0,FnB,보조적,0.538,0.854,0.316
1,FnB,없음,0.019,0.021,0.002
2,FnB,핵심요소,0.442,0.125,-0.317
4,IT,핵심요소,0.276,0.548,0.272
3,IT,보조적,0.724,0.452,-0.272


editing_pace


success_label,domain,editing_pace,fail,success,diff_success_minus_fail
3,FnB,빠름,0.462,0.625,0.163
0,FnB,매우 느림,0.038,0.021,-0.017
2,FnB,보통,0.115,0.083,-0.032
1,FnB,매우 빠름,0.385,0.271,-0.114
6,IT,매우 빠름,0.172,0.262,0.090
8,IT,빠름,0.466,0.548,0.082
4,IT,느림,0.017,0.000,-0.017
5,IT,매우 느림,0.069,0.000,-0.069
7,IT,보통,0.276,0.190,-0.086


video_format


success_label,domain,video_format,fail,success,diff_success_minus_fail
7,FnB,웹예능,0.019,0.167,0.148
10,FnB,제품리뷰,0.019,0.083,0.064
6,FnB,웹드라마,0.000,0.062,0.062
8,FnB,이벤트/행사,0.077,0.104,0.027
2,FnB,시설소개,0.019,0.021,0.002
4,FnB,에피소드소개,0.019,0.021,0.002
3,FnB,애니메이션,0.038,0.021,-0.017
9,FnB,인터뷰,0.038,0.021,-0.017
1,FnB,브이로그,0.019,0.000,-0.019
5,FnB,요리/레시피,0.019,0.000,-0.019


# Cell 19. 전체 범주형 차이표 합치기

In [23]:
category_diff_all = []

for feature, table in category_diff_dict.items():
    temp = table.copy()
    temp["feature"] = feature
    temp["category"] = temp[feature]
    
    category_diff_all.append(
        temp[["domain", "feature", "category", "success", "fail", "diff_success_minus_fail"]]
    )

category_diff_all_df = pd.concat(category_diff_all, ignore_index=True)

display(
    category_diff_all_df.sort_values(
        ["domain", "feature", "diff_success_minus_fail"],
        ascending=[True, True, False]
    )
)

success_label,domain,feature,category,success,fail,diff_success_minus_fail
15,FnB,editing_pace,빠름,0.625,0.462,0.163
16,FnB,editing_pace,매우 느림,0.021,0.038,-0.017
17,FnB,editing_pace,보통,0.083,0.115,-0.032
18,FnB,editing_pace,매우 빠름,0.271,0.385,-0.114
0,FnB,first_3sec,인물,0.521,0.212,0.309
1,FnB,first_3sec,장면,0.042,0.019,0.023
2,FnB,first_3sec,제품,0.167,0.173,-0.006
3,FnB,first_3sec,음식,0.021,0.038,-0.017
4,FnB,first_3sec,기업 로고,0.000,0.019,-0.019
5,FnB,first_3sec,텍스트,0.250,0.538,-0.288


# Cell 20. 인사이트 1: FnB 인물/얼굴 중심 구성 확인

In [24]:
print("FnB 수치형 핵심 변수")
display(
    numeric_test_with_assumption_df[
        (numeric_test_with_assumption_df["domain"] == "FnB") &
        (numeric_test_with_assumption_df["feature"].isin(["person_ratio", "face_ratio"]))
    ]
)

print("FnB first_3sec 검정")
display(
    category_test_df[
        (category_test_df["domain"] == "FnB") &
        (category_test_df["feature"] == "first_3sec")
    ]
)

print("FnB first_3sec 범주별 차이")
display(
    category_diff_dict["first_3sec"][
        category_diff_dict["first_3sec"]["domain"] == "FnB"
    ]
)

FnB 수치형 핵심 변수


,domain,feature,success_mean,fail_mean,diff_success_minus_fail,t_stat,t_test_p,u_stat,mannwhitney_p,cohens_d,success_n,fail_n,t_test_p_fdr,mannwhitney_p_fdr,shapiro_p_success,normality_result_success,shapiro_p_fail,normality_result_fail,levene_p,variance_result
4,FnB,person_ratio,0.786,0.602,0.185,3.0011,0.0034,1734.0,0.0008,0.595,48,52,0.0119,0.0032,0.0,정규성 불만족 가능,0.0,정규성 불만족 가능,0.0367,등분산 가정 어려움
5,FnB,face_ratio,0.649,0.399,0.250,3.6647,0.0004,1724.5,0.0009,0.727,48,52,0.0028,0.0032,0.0,정규성 불만족 가능,0.0,정규성 불만족 가능,0.0009,등분산 가정 어려움


FnB first_3sec 검정


,domain,feature,chi2_stat,chi2_p,dof,cramers_v,n_categories,success_n,fail_n,min_expected,low_expected_count,low_expected_ratio,expected_freq_warning,chi2_p_fdr
6,FnB,first_3sec,13.4314,0.0197,5,0.366,6,48,52,0.48,6,0.5,주의 필요,0.0788


FnB first_3sec 범주별 차이


success_label,domain,first_3sec,fail,success,diff_success_minus_fail
2,FnB,인물,0.212,0.521,0.309
3,FnB,장면,0.019,0.042,0.023
4,FnB,제품,0.173,0.167,-0.006
1,FnB,음식,0.038,0.021,-0.017
0,FnB,기업 로고,0.019,0.000,-0.019
5,FnB,텍스트,0.538,0.250,-0.288


# Cell 21. 인사이트 2: FnB 모션그래픽 보조 활용 확인

In [25]:
print("FnB motion_graphic 검정")
display(
    category_test_df[
        (category_test_df["domain"] == "FnB") &
        (category_test_df["feature"] == "motion_graphic")
    ]
)

print("FnB motion_graphic 범주별 차이")
display(
    category_diff_dict["motion_graphic"][
        category_diff_dict["motion_graphic"]["domain"] == "FnB"
    ]
)

FnB motion_graphic 검정


,domain,feature,chi2_stat,chi2_p,dof,cramers_v,n_categories,success_n,fail_n,min_expected,low_expected_count,low_expected_ratio,expected_freq_warning,chi2_p_fdr
4,FnB,motion_graphic,12.2744,0.0022,2,0.35,3,48,52,0.96,2,0.333,주의 필요,0.0176


FnB motion_graphic 범주별 차이


success_label,domain,motion_graphic,fail,success,diff_success_minus_fail
0,FnB,보조적,0.538,0.854,0.316
1,FnB,없음,0.019,0.021,0.002
2,FnB,핵심요소,0.442,0.125,-0.317


# Cell 22. 인사이트 3: IT 모션그래픽 / 첫 3초 텍스트 확인

In [26]:
print("IT text_ratio 검정")
display(
    numeric_test_with_assumption_df[
        (numeric_test_with_assumption_df["domain"] == "IT") &
        (numeric_test_with_assumption_df["feature"] == "text_ratio")
    ]
)

print("IT first_3sec, motion_graphic 검정")
display(
    category_test_df[
        (category_test_df["domain"] == "IT") &
        (category_test_df["feature"].isin(["first_3sec", "motion_graphic"]))
    ]
)

print("IT first_3sec 범주별 차이")
display(
    category_diff_dict["first_3sec"][
        category_diff_dict["first_3sec"]["domain"] == "IT"
    ]
)

print("IT motion_graphic 범주별 차이")
display(
    category_diff_dict["motion_graphic"][
        category_diff_dict["motion_graphic"]["domain"] == "IT"
    ]
)

IT text_ratio 검정


,domain,feature,success_mean,fail_mean,diff_success_minus_fail,t_stat,t_test_p,u_stat,mannwhitney_p,cohens_d,success_n,fail_n,t_test_p_fdr,mannwhitney_p_fdr,shapiro_p_success,normality_result_success,shapiro_p_fail,normality_result_fail,levene_p,variance_result
13,IT,text_ratio,0.807,0.769,0.037,1.2021,0.2322,1334.5,0.4118,0.233,42,58,0.2322,0.4118,0.0091,정규성 불만족 가능,0.0006,정규성 불만족 가능,0.1061,등분산 가정 가능


IT first_3sec, motion_graphic 검정


,domain,feature,chi2_stat,chi2_p,dof,cramers_v,n_categories,success_n,fail_n,min_expected,low_expected_count,low_expected_ratio,expected_freq_warning,chi2_p_fdr
12,IT,motion_graphic,6.4630,0.0110,1,0.254,2,42,58,16.38,0,0.0,대체로 양호,0.0880
14,IT,first_3sec,6.7947,0.0787,3,0.261,4,42,58,0.84,4,0.5,주의 필요,0.2712


IT first_3sec 범주별 차이


success_label,domain,first_3sec,fail,success,diff_success_minus_fail
9,IT,텍스트,0.621,0.833,0.212
8,IT,제품,0.034,0.000,-0.034
7,IT,장면,0.052,0.000,-0.052
6,IT,인물,0.293,0.167,-0.126


IT motion_graphic 범주별 차이


success_label,domain,motion_graphic,fail,success,diff_success_minus_fail
4,IT,핵심요소,0.276,0.548,0.272
3,IT,보조적,0.724,0.452,-0.272


# Cell 23. 인사이트 4: 편집 속도 확인

In [27]:
print("editing_pace 검정")
display(
    category_test_df[
        category_test_df["feature"] == "editing_pace"
    ]
)

print("editing_pace 범주별 차이")
display(category_diff_dict["editing_pace"])

editing_pace 검정


,domain,feature,chi2_stat,chi2_p,dof,cramers_v,n_categories,success_n,fail_n,min_expected,low_expected_count,low_expected_ratio,expected_freq_warning,chi2_p_fdr
3,FnB,editing_pace,2.7292,0.4353,3,0.165,4,48,52,1.44,3,0.375,주의 필요,0.5855
11,IT,editing_pace,5.6181,0.2295,4,0.237,5,42,58,0.42,4,0.400,주의 필요,0.3672


editing_pace 범주별 차이


success_label,domain,editing_pace,fail,success,diff_success_minus_fail
3,FnB,빠름,0.462,0.625,0.163
0,FnB,매우 느림,0.038,0.021,-0.017
2,FnB,보통,0.115,0.083,-0.032
1,FnB,매우 빠름,0.385,0.271,-0.114
6,IT,매우 빠름,0.172,0.262,0.090
8,IT,빠름,0.466,0.548,0.082
4,IT,느림,0.017,0.000,-0.017
5,IT,매우 느림,0.069,0.000,-0.069
7,IT,보통,0.276,0.190,-0.086


# Cell 24. 인사이트 5: 영상 포맷 확인

In [28]:
print("video_format 검정")
display(
    category_test_df[
        category_test_df["feature"] == "video_format"
    ]
)

print("video_format 범주별 차이")
display(category_diff_dict["video_format"])

video_format 검정


,domain,feature,chi2_stat,chi2_p,dof,cramers_v,n_categories,success_n,fail_n,min_expected,low_expected_count,low_expected_ratio,expected_freq_warning,chi2_p_fdr
5,FnB,video_format,16.7500,0.1155,11,0.409,12,48,52,0.48,20,0.833,주의 필요,0.3080
13,IT,video_format,18.4849,0.1017,12,0.430,13,42,58,0.42,19,0.731,주의 필요,0.2712


video_format 범주별 차이


success_label,domain,video_format,fail,success,diff_success_minus_fail
7,FnB,웹예능,0.019,0.167,0.148
10,FnB,제품리뷰,0.019,0.083,0.064
6,FnB,웹드라마,0.000,0.062,0.062
8,FnB,이벤트/행사,0.077,0.104,0.027
2,FnB,시설소개,0.019,0.021,0.002
4,FnB,에피소드소개,0.019,0.021,0.002
3,FnB,애니메이션,0.038,0.021,-0.017
9,FnB,인터뷰,0.038,0.021,-0.017
1,FnB,브이로그,0.019,0.000,-0.019
5,FnB,요리/레시피,0.019,0.000,-0.019


# Cell 25. FigJam/PPT용 핵심 통계 요약표

In [30]:
summary_rows = []

# FnB person_ratio
fnb_person = numeric_test_with_assumption_df[
    (numeric_test_with_assumption_df["domain"] == "FnB") &
    (numeric_test_with_assumption_df["feature"] == "person_ratio")
].iloc[0]

summary_rows.append({
    "insight": "FnB 성공 숏츠는 인물 등장 비율이 높음",
    "domain": "FnB",
    "test_type": "Welch t-test / Mann-Whitney",
    "feature": "person_ratio",
    "result": f"성공 {fnb_person['success_mean']} / 실패 {fnb_person['fail_mean']} / 차이 {fnb_person['diff_success_minus_fail']}",
    "p_value": f"t={fnb_person['t_test_p']}, MW={fnb_person['mannwhitney_p']}",
    "effect_size": f"Cohen's d={fnb_person['cohens_d']}",
    "interpretation": "성공 숏츠에서 인물 비율이 통계적으로 높게 나타남"
})

# FnB face_ratio
fnb_face = numeric_test_with_assumption_df[
    (numeric_test_with_assumption_df["domain"] == "FnB") &
    (numeric_test_with_assumption_df["feature"] == "face_ratio")
].iloc[0]

summary_rows.append({
    "insight": "FnB 성공 숏츠는 얼굴 등장 비율이 높음",
    "domain": "FnB",
    "test_type": "Welch t-test / Mann-Whitney",
    "feature": "face_ratio",
    "result": f"성공 {fnb_face['success_mean']} / 실패 {fnb_face['fail_mean']} / 차이 {fnb_face['diff_success_minus_fail']}",
    "p_value": f"t={fnb_face['t_test_p']}, MW={fnb_face['mannwhitney_p']}",
    "effect_size": f"Cohen's d={fnb_face['cohens_d']}",
    "interpretation": "성공 숏츠에서 얼굴과 반응 장면이 더 두드러짐"
})

# FnB first_3sec
fnb_first = category_test_df[
    (category_test_df["domain"] == "FnB") &
    (category_test_df["feature"] == "first_3sec")
].iloc[0]

summary_rows.append({
    "insight": "FnB 성공 숏츠는 첫 3초에 인물 등장 비율이 높음",
    "domain": "FnB",
    "test_type": "Chi-square",
    "feature": "first_3sec",
    "result": "first_3sec=인물 성공 52.1% / 실패 21.2%",
    "p_value": f"chi2 p={fnb_first['chi2_p']}",
    "effect_size": f"Cramer's V={fnb_first['cramers_v']}",
    "interpretation": "FnB는 첫 3초에 인물 경험 장면을 배치하는 전략이 중요해 보임"
})

# FnB motion_graphic
fnb_motion = category_test_df[
    (category_test_df["domain"] == "FnB") &
    (category_test_df["feature"] == "motion_graphic")
].iloc[0]

summary_rows.append({
    "insight": "FnB 성공 숏츠는 모션그래픽을 보조적으로 활용",
    "domain": "FnB",
    "test_type": "Chi-square",
    "feature": "motion_graphic",
    "result": "보조적 활용 성공 85.4% / 실패 53.8%",
    "p_value": f"chi2 p={fnb_motion['chi2_p']}",
    "effect_size": f"Cramer's V={fnb_motion['cramers_v']}",
    "interpretation": "FnB는 그래픽보다 제품 경험 장면이 중심이 되는 구성이 적합"
})

# IT motion_graphic
it_motion = category_test_df[
    (category_test_df["domain"] == "IT") &
    (category_test_df["feature"] == "motion_graphic")
].iloc[0]

summary_rows.append({
    "insight": "IT 성공 숏츠는 모션그래픽을 핵심 요소로 활용",
    "domain": "IT",
    "test_type": "Chi-square",
    "feature": "motion_graphic",
    "result": "핵심요소 성공 54.8% / 실패 27.6%",
    "p_value": f"chi2 p={it_motion['chi2_p']}",
    "effect_size": f"Cramer's V={it_motion['cramers_v']}",
    "interpretation": "IT는 기능·서비스·기술 정보를 시각화하는 구성이 중요해 보임"
})

# IT first_3sec
it_first = category_test_df[
    (category_test_df["domain"] == "IT") &
    (category_test_df["feature"] == "first_3sec")
].iloc[0]

summary_rows.append({
    "insight": "IT 성공 숏츠는 첫 3초 텍스트 도입 경향",
    "domain": "IT",
    "test_type": "Chi-square",
    "feature": "first_3sec",
    "result": "first_3sec=텍스트 성공 83.3% / 실패 62.1%",
    "p_value": f"chi2 p={it_first['chi2_p']}",
    "effect_size": f"Cramer's V={it_first['cramers_v']}",
    "interpretation": "통계적으로 강하게 확정되지는 않았지만, 텍스트 후킹 경향이 관찰됨"
})

stat_insight_summary_df = pd.DataFrame(summary_rows)

display(stat_insight_summary_df)

,insight,domain,test_type,feature,result,p_value,effect_size,interpretation
0,FnB 성공 숏츠는 인물 등장 비율이 높음,FnB,Welch t-test / Mann-Whitney,person_ratio,성공 0.786 / 실패 0.602 / 차이 0.185,"t=0.0034, MW=0.0008",Cohen's d=0.595,성공 숏츠에서 인물 비율이 통계적으로 높게 나타남
1,FnB 성공 숏츠는 얼굴 등장 비율이 높음,FnB,Welch t-test / Mann-Whitney,face_ratio,성공 0.649 / 실패 0.399 / 차이 0.25,"t=0.0004, MW=0.0009",Cohen's d=0.727,성공 숏츠에서 얼굴과 반응 장면이 더 두드러짐
2,FnB 성공 숏츠는 첫 3초에 인물 등장 비율이 높음,FnB,Chi-square,first_3sec,first_3sec=인물 성공 52.1% / 실패 21.2%,chi2 p=0.0197,Cramer's V=0.366,FnB는 첫 3초에 인물 경험 장면을 배치하는 전략이 중요해 보임
3,FnB 성공 숏츠는 모션그래픽을 보조적으로 활용,FnB,Chi-square,motion_graphic,보조적 활용 성공 85.4% / 실패 53.8%,chi2 p=0.0022,Cramer's V=0.35,FnB는 그래픽보다 제품 경험 장면이 중심이 되는 구성이 적합
4,IT 성공 숏츠는 모션그래픽을 핵심 요소로 활용,IT,Chi-square,motion_graphic,핵심요소 성공 54.8% / 실패 27.6%,chi2 p=0.011,Cramer's V=0.254,IT는 기능·서비스·기술 정보를 시각화하는 구성이 중요해 보임
5,IT 성공 숏츠는 첫 3초 텍스트 도입 경향,IT,Chi-square,first_3sec,first_3sec=텍스트 성공 83.3% / 실패 62.1%,chi2 p=0.0787,Cramer's V=0.261,"통계적으로 강하게 확정되지는 않았지만, 텍스트 후킹 경향이 관찰됨"


In [31]:
method_summary = """
[통계 검정 방법론 요약]

수치형 변수에 대해 Shapiro-Wilk test로 정규성을 확인하고, Levene test로 등분산성을 확인하였다.
일부 변수는 정규성 또는 등분산성 가정을 만족하지 않을 가능성이 있어,
평균 차이 검정에는 등분산을 가정하지 않는 Welch’s t-test를 사용하였다.
또한 정규성 가정에 덜 민감한 Mann-Whitney U test를 함께 수행하여 결과를 보완하였다.
효과크기는 Cohen’s d를 사용해 평균 차이의 크기를 함께 해석하였다.

범주형 변수는 성공/실패와 각 범주형 특징 간의 분포 차이를 확인하기 위해 카이제곱 독립성 검정을 사용하였다.
관계의 강도는 Cramer’s V로 확인하였다.
또한 기대빈도를 점검하여, 범주 수가 많고 기대빈도가 낮은 변수는 보조 인사이트로 해석하였다.

여러 변수를 동시에 검정했기 때문에 p-value는 탐색적 근거로 해석하고,
Benjamini-Hochberg FDR 보정 결과를 함께 참고하였다.
"""

print(method_summary)


[통계 검정 방법론 요약]

수치형 변수에 대해 Shapiro-Wilk test로 정규성을 확인하고, Levene test로 등분산성을 확인하였다.
일부 변수는 정규성 또는 등분산성 가정을 만족하지 않을 가능성이 있어,
평균 차이 검정에는 등분산을 가정하지 않는 Welch’s t-test를 사용하였다.
또한 정규성 가정에 덜 민감한 Mann-Whitney U test를 함께 수행하여 결과를 보완하였다.
효과크기는 Cohen’s d를 사용해 평균 차이의 크기를 함께 해석하였다.

범주형 변수는 성공/실패와 각 범주형 특징 간의 분포 차이를 확인하기 위해 카이제곱 독립성 검정을 사용하였다.
관계의 강도는 Cramer’s V로 확인하였다.
또한 기대빈도를 점검하여, 범주 수가 많고 기대빈도가 낮은 변수는 보조 인사이트로 해석하였다.

여러 변수를 동시에 검정했기 때문에 p-value는 탐색적 근거로 해석하고,
Benjamini-Hochberg FDR 보정 결과를 함께 참고하였다.



In [32]:
OUTPUT_DIR = Path("eda_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

normality_df.to_csv(
    OUTPUT_DIR / "shorts_numeric_normality_test.csv",
    index=False,
    encoding="utf-8-sig"
)

variance_df.to_csv(
    OUTPUT_DIR / "shorts_numeric_variance_test.csv",
    index=False,
    encoding="utf-8-sig"
)

assumption_summary.to_csv(
    OUTPUT_DIR / "shorts_numeric_assumption_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

numeric_test_df.to_csv(
    OUTPUT_DIR / "shorts_numeric_stat_test.csv",
    index=False,
    encoding="utf-8-sig"
)

numeric_test_with_assumption_df.to_csv(
    OUTPUT_DIR / "shorts_numeric_stat_test_with_assumption.csv",
    index=False,
    encoding="utf-8-sig"
)

category_test_df.to_csv(
    OUTPUT_DIR / "shorts_category_chi_square_test.csv",
    index=False,
    encoding="utf-8-sig"
)

category_diff_all_df.to_csv(
    OUTPUT_DIR / "shorts_category_success_fail_diff.csv",
    index=False,
    encoding="utf-8-sig"
)

stat_insight_summary_df.to_csv(
    OUTPUT_DIR / "shorts_stat_insight_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", OUTPUT_DIR.resolve())

저장 완료: C:\Users\user\Desktop\Uk_folder\sparta_bootcamp\project_collection\final_project\final_project\works\Hyeong_Uk\shorts_video_analysis\eda_outputs
